# HDB Resale Flat Prices ETL — Part 1

Part 1 of the HDB Senior Data Engineer technical test. ETL pipeline producing cleaned, transformed, and hashed datasets of HDB resale flat transactions covering **January 2012 – December 2016**, sourced from [data.gov.sg collection 189](https://data.gov.sg/collections/189/view).

## How to use this notebook

Run the cells top to bottom. On a fresh clone the raw CSVs are already present in `data/raw/`, so the pipeline runs end-to-end without network access. The download path is still exercised when run, and is idempotent: files whose size matches the API's `datasetSize` are skipped.

All pipeline logic lives in `src/`. The notebook imports it and orchestrates each stage; heavy logic is intentionally kept out of cells.

## Stages mapped to the brief

| Brief requirement | Notebook section | Status |
|---|---|---|
| Dataset extraction (Jan 2012 – Dec 2016) | §1 Ingest | implemented |
| Combine into a single master dataset (DQ §1) | §2 Combine | implemented |
| Data profiling (DQ §2) | §3 Profile | implemented |
| Validation rules on Date/Town/Flat Type/Flat Model/storey_range (DQ §3) | §4 Validate | implemented |
| Recompute `remaining_lease` as of today (DQ §4) | §5 Clean | implemented |
| Dedupe on composite key, keep highest price (DQ §5) | §5 Clean | implemented |
| Anomaly heuristic on resale price (DQ §6) | §5 Clean | implemented |
| Resale Identifier (Transformation §1) | §6 Transform | implemented |
| Dedupe on duplicate identifiers (Transformation §2) | §6 Transform | implemented |
| Irreversible hashed identifier (Transformation §3) | §6 Transform | implemented |

## 0. Setup

Configure logging and import the pipeline modules. Editable install (`pip install -e ".[dev]"`) makes `from src...` resolvable from the notebook working directory.

In [1]:
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import pyarrow
import requests

# Environment sanity check 
print(f"python:   {sys.version.split()[0]}")
print(f"pandas:   {pd.__version__}")
print(f"requests: {requests.__version__}")
print(f"pyarrow:  {pyarrow.__version__}")

# INFO-level surfaces the per-stage progress lines emitted by src/ingest.py
# and src/combine.py directly into the notebook output.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

from src import config
from src.ingest import ingest_all, verify_raw
from src.combine import combine_raw_files

print()
print("project root:", config.PROJECT_ROOT)
print("scope window:", config.SCOPE_START, "..", config.SCOPE_END)
print("as_of_date:  ", config.AS_OF_DATE)

python:   3.12.2
pandas:   3.0.2
requests: 2.33.1
pyarrow:  23.0.1

project root: /Users/ivanong/Documents/GitHub/hdb_resale_prices
scope window: 2012-01 .. 2016-12
as_of_date:   2026-04-09


## 1. Ingest

**Brief reference:** *Dataset Extraction* — *"Your ETL pipeline should process the data file as it is, without any manual modifications."*

**Goal:** Download the in-scope HDB resale CSVs from data.gov.sg into `data/raw/`.

**Approach:** Discover dataset IDs at runtime by walking the collection metadata endpoint and filtering on coverage window — no dataset IDs or filenames are hardcoded, in line with the brief's *"avoid hardcoding where possible"* directive. Downloads stream from the pre-signed S3 URLs returned by the `poll-download` endpoint, with 429 retry/backoff and `datasetSize`-based idempotency.

**Robustness:** If the API is unreachable, the pipeline falls back to whatever CSVs are already present in `data/raw/`. Because raw data is committed to the repo, the notebook is runnable on a fresh clone with no network access.

In [2]:
downloaded_paths = ingest_all()

print()
print(f"{len(downloaded_paths)} files in data/raw/:")
for p in downloaded_paths:
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

2026-04-09 14:51:27,057 INFO src.ingest: Collection 189 has 5 child datasets


2026-04-09 14:51:30,062 INFO src.ingest: Filtered to 3 in-scope datasets for 2012-01..2016-12


2026-04-09 14:51:30,319 INFO src.ingest: poll-download rate-limited for d_43f493c6c50d54243cc1eab0df142d6a; sleeping 3.0s


2026-04-09 14:51:33,531 INFO src.ingest: poll-download rate-limited for d_43f493c6c50d54243cc1eab0df142d6a; sleeping 3.0s


2026-04-09 14:51:36,727 INFO src.ingest: poll-download rate-limited for d_43f493c6c50d54243cc1eab0df142d6a; sleeping 3.0s


...

2026-04-09 14:51:54,363 INFO src.ingest: poll-download rate-limited for d_ea9ed51da2787afaf8e51f827c304208; sleeping 3.0s


2026-04-09 14:51:57,567 INFO src.ingest: poll-download rate-limited for d_ea9ed51da2787afaf8e51f827c304208; sleeping 3.0s


2026-04-09 14:52:00,800 INFO src.ingest: poll-download rate-limited for d_ea9ed51da2787afaf8e51f827c304208; sleeping 3.0s


2026-04-09 14:52:04,226 INFO src.ingest: Skip ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv (already present, 3070924 bytes)



3 files in data/raw/:
  ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv  (29,369,945 bytes)
  ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv  (4,160,771 bytes)
  ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv  (3,070,924 bytes)


### 1.1 Verify

For each CSV in `data/raw/`, parse it with pandas and check three things against the API's authoritative metadata:

| Check | What it confirms |
|---|---|
| **size_ok** | File size equals the API's `datasetSize` exactly |
| **cols_ok** | Column names equal `columnMetadata` (in order) |
| **dates_ok** | Every row's `month` falls inside the dataset's coverage window |

Together these form the strongest integrity envelope available without a server-published checksum. `verify_raw()` also reports any in-scope dataset that has no local file, so the inverse failure mode is caught too.

In [3]:
verification = verify_raw()
verification_df = pd.DataFrame(verification)

display(verification_df[[
    "file", "matched_dataset", "rows", "cols",
    "min_month", "max_month",
    "actual_size", "expected_size",
    "size_ok", "cols_ok", "dates_ok", "ok",
]])

assert all(r["ok"] for r in verification), "Stage 1 verification failed — see logs above"

2026-04-09 14:52:04,462 INFO src.ingest: Collection 189 has 5 child datasets


2026-04-09 14:52:06,228 INFO src.ingest: Filtered to 3 in-scope datasets for 2012-01..2016-12


2026-04-09 14:52:06,506 INFO src.ingest: [OK] ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv  rows=369651 cols=10  2000-01..2012-02  size_ok=True cols_ok=True dates_ok=True


2026-04-09 14:52:06,538 INFO src.ingest: [OK] ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv  rows=37153 cols=11  2015-01..2016-12  size_ok=True cols_ok=True dates_ok=True


2026-04-09 14:52:06,581 INFO src.ingest: [OK] ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv  rows=52203 cols=10  2012-03..2014-12  size_ok=True cols_ok=True dates_ok=True


2026-04-09 14:52:06,581 INFO src.ingest: verify_raw: 3 files checked, overall OK


,file,matched_dataset,rows,cols,min_month,max_month,actual_size,expected_size,size_ok,cols_ok,dates_ok,ok
0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,d_43f493c6c50d54243cc1eab0df142d6a,369651,10,2000-01,2012-02,29369945,29369945,True,True,True,True
1,ResaleFlatPricesBasedonRegistrationDateFromJan...,d_ea9ed51da2787afaf8e51f827c304208,37153,11,2015-01,2016-12,3070924,3070924,True,True,True,True
2,ResaleFlatPricesBasedonRegistrationDateFromMar...,d_2d5ff9ea31397b66239f245f57751537,52203,10,2012-03,2014-12,4160771,4160771,True,True,True,True


## 2. Combine

**Brief reference:** *Data Quality Requirements §1* — *"Combine the datasets into a single master dataset. The combined dataset should contain all the attributes found in all files."*

**Goal:** Union the verified raw CSVs into a single master DataFrame ready for profiling, validation, and cleaning.

**Per-file pipeline:**

1. **Normalize headers** — `strip().lower().replace(" ", "_")` on every column on read. Defends against case/whitespace drift across vintages so the schema union doesn't produce duplicates like `Month` vs `month`.
2. **Filter to scope** — drop rows whose `month` falls outside `SCOPE_START..SCOPE_END`. For the 2000–Feb 2012 file this trims to its Jan–Feb 2012 tail; for the other two it's a no-op. Applying it uniformly avoids special-casing.
3. **Normalize `remaining_lease`** — the Jan 2015–Dec 2016 file stores it as integer years already (verified by inspection: dtype `int64`, sample values like 70, 65, 64). We copy it into a canonical `remaining_lease_years_original` `Int64` column. Pre-2015 vintages don't have this column at all, so the master will hold `NaN` for those rows. The 2017+ "X years Y months" string format is out of scope and not parsed here.
4. **Tag with `source_file`** — lineage column for downstream debugging.

Frames are then concatenated with `sort=False`, taking the union of columns (pandas fills `NaN` where a column is absent in any particular frame).

In [4]:
master = combine_raw_files()

print()
print(f"Master shape: {master.shape[0]:,} rows x {master.shape[1]} columns")
print()
print("Rows per source file:")
print(master.groupby("source_file").size().to_string())
print()
print(f"Month range: {master['month'].min()} .. {master['month'].max()}")
print()
print("remaining_lease_years_original by source file:")
print(
    master.groupby("source_file")["remaining_lease_years_original"]
    .apply(lambda s: f"{s.notna().sum():>6,d} / {len(s):>6,d} populated")
    .to_string()
)

2026-04-09 14:52:06,845 INFO src.combine: ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv: 369651 rows raw, 3188 in scope (2012-01..2016-12)


2026-04-09 14:52:06,877 INFO src.combine: ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv: 37153 rows raw, 37153 in scope (2012-01..2016-12)


2026-04-09 14:52:06,918 INFO src.combine: ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv: 52203 rows raw, 52203 in scope (2012-01..2016-12)


2026-04-09 14:52:06,920 INFO src.combine: Combined master: 92544 rows, 13 columns from 3 files



Master shape: 92,544 rows x 13 columns

Rows per source file:
source_file
ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv                  3188
ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv    37153
ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv    52203

Month range: 2012-01 .. 2016-12

remaining_lease_years_original by source file:
source_file
ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv                      0 /  3,188 populated
ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv    37,153 / 37,153 populated
ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv         0 / 52,203 populated


### 2.1 Sample

First five rows of the combined master, with all 13 columns:

In [5]:
display(master.head())

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,remaining_lease,remaining_lease_years_original
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>


## 3. Profile

**Brief reference:** *Data Quality Requirements §2* — *"Perform data profiling on the master dataset."*

**Goal:** Surface the structural and statistical properties of the master so the next stages (validation rules, cleaning policies, anomaly thresholds) are informed by what's actually in the data, not assumed.

**Approach:** Hand-rolled profiling in `src/profile.py` (no `ydata-profiling`). The output is a `ProfileReport` dataclass holding pre-built DataFrames so each subsection below renders with a single `display(...)` call. Sections:

* **3.1 Overview** — per-column dtype/null/unique/sample
* **3.2 Numeric stats** — min/max/mean/median/std/quartiles
* **3.3 String columns** — length distribution and top-N value counts
* **3.4 Flagged columns** — high-null, constant, and whitespace contamination

The function is pure — it does not mutate the master.

In [6]:
from src.profile import profile_dataset

report = profile_dataset(master)

print(f"Rows x cols: {report.shape[0]:,} x {report.shape[1]}")
print(f"Month range: {report.month_range[0]} .. {report.month_range[1]}")
print(f"Numeric columns:    {len(report.numeric)}")
print(f"String columns:     {len(report.string_lengths)} (excluding source_file)")
print(f"Flagged columns:    {report.flags['column'].nunique() if not report.flags.empty else 0}")

Rows x cols: 92,544 x 13
Month range: 2012-01 .. 2016-12
Numeric columns:    5
String columns:     7 (excluding source_file)
Flagged columns:    2


### 3.1 Per-column overview

One row per column: dtype, null count and percentage, distinct value count, and a short sample of distinct non-null values. The `unique_count` here is what Stage 4 will use to size each rule's allowed set.

In [7]:
display(report.overview)

,column,dtype,null_count,null_pct,unique_count,sample_values
0,month,str,0,0.000000,60,"'2012-01', '2012-02', '2015-01'"
1,town,str,0,0.000000,26,"'ANG MO KIO', 'BEDOK', 'BISHAN'"
2,flat_type,str,0,0.000000,7,"'2 ROOM', '3 ROOM', '4 ROOM'"
3,block,str,0,0.000000,2139,"'406', '314', '170'"
4,street_name,str,0,0.000000,522,"'ANG MO KIO AVE 10', 'ANG MO KIO AVE 3', 'ANG ..."
5,storey_range,str,0,0.000000,25,"'01 TO 03', '07 TO 09', '10 TO 12'"
6,floor_area_sqm,float64,0,0.000000,168,"44.0, 45.0, 60.0"
7,flat_model,str,0,0.000000,20,"'Improved', 'New Generation', 'Model A'"
8,lease_commence_date,int64,0,0.000000,48,"1979, 1978, 1986"
9,resale_price,float64,0,0.000000,2615,"257800.0, 263000.0, 275000.0"


### 3.2 Numeric column statistics

`min`, `max`, `mean`, `median`, `std`, and quartiles for every numeric column. These feed Stage 5's anomaly heuristic (asymmetric IQR on price-per-sqm) and let us sanity-check that `floor_area_sqm`, `lease_commence_date`, and `resale_price` are within plausible HDB ranges.

In [8]:
display(report.numeric)

,min,max,mean,median,std,q1,q3
floor_area_sqm,31.0,280.0,96.569115,95.0,24.682292,74.0,111.0
lease_commence_date,1966.0,2013.0,1990.072701,1988.0,10.446719,1983.0,1999.0
resale_price,190000.0,1150000.0,450938.971730,428000.0,128181.301162,357000.0,515000.0
remaining_lease,48.0,97.0,73.913116,72.0,10.885456,66.0,83.0
remaining_lease_years_original,48.0,97.0,73.913116,72.0,10.885456,66.0,83.0


### 3.3 String columns — lengths and top values

Length stats catch unexpectedly long or zero-length values; top-N value counts give a feel for cardinality and the dominant categories that Stage 4's validation rules will derive their allowed sets from.

In [9]:
display(report.string_lengths)

# Top-N value counts per string column. `source_file` is excluded by design
# (lineage metadata, has exactly one value per source file).
for col, vc in report.string_top_values.items():
    print(f"\n--- top {len(vc)} values: {col} ---")
    display(vc)

,column,min_len,max_len,mean_len
0,month,7,7,7.000000
1,town,5,15,9.095468
2,flat_type,6,16,6.239400
3,block,1,4,2.995278
4,street_name,7,20,14.055746
5,storey_range,8,8,8.000000
6,flat_model,4,22,9.813300



--- top 10 values: month ---


,month,count
0,2012-03,2360
1,2012-05,2323
2,2012-07,2179
3,2012-04,2155
4,2012-08,2075
5,2012-06,1993
6,2012-10,1937
7,2016-08,1882
8,2016-05,1829
9,2016-06,1827



--- top 10 values: town ---


,town,count
0,JURONG WEST,7573
1,WOODLANDS,7399
2,TAMPINES,6728
3,BEDOK,6071
4,YISHUN,5937
5,SENGKANG,5896
6,HOUGANG,4735
7,ANG MO KIO,4558
8,CHOA CHU KANG,3928
9,BUKIT BATOK,3773


...

,street_name,count
0,YISHUN RING RD,1629
1,BEDOK RESERVOIR RD,1264
2,ANG MO KIO AVE 10,1188
3,ANG MO KIO AVE 3,1059
4,HOUGANG AVE 8,844
5,JURONG WEST ST 65,777
6,CHOA CHU KANG CRES,737
7,BEDOK NTH RD,731
8,BEDOK NTH ST 3,708
9,PUNGGOL CTRL,644



--- top 10 values: storey_range ---


,storey_range,count
0,04 TO 06,21214
1,07 TO 09,18765
2,01 TO 03,17466
3,10 TO 12,16028
4,13 TO 15,6704
5,16 TO 18,2706
6,01 TO 05,2700
7,06 TO 10,2474
8,11 TO 15,1259
9,19 TO 21,1156



--- top 10 values: flat_model ---


,flat_model,count
0,Model A,26447
1,Improved,24117
2,New Generation,16495
3,Premium Apartment,8314
4,Simplified,5152
5,Apartment,3701
6,Standard,3458
7,Maisonette,2574
8,Model A2,1415
9,DBSS,277


### 3.4 Flagged columns

Three flag types are emitted (config in `src/config.py`):

* **`high_null`** — null fraction > `PROFILE_HIGH_NULL_THRESHOLD` (50%).
* **`constant`** — only one distinct non-null value.
* **`whitespace`** — at least one string value differs from its `.str.strip()` form.

**Expected finding on this master:** only `remaining_lease` and `remaining_lease_years_original` should appear, both flagged `high_null` at ~60%. This is the vintage gap, not a defect — the pre-2015 source files don't carry the column at all (see the schema-by-vintage table in `README.md`). Stage 4 (validate) will treat this as known-absent rather than failing rows on it.

In [10]:
display(report.flags)

,column,flag,detail
0,remaining_lease,high_null,59.9% nulls
1,remaining_lease_years_original,high_null,59.9% nulls


## 4. Validate

**Brief reference:** *Data Quality Requirements §3* — *"Document and apply data validation rules for at least the following columns: Date, Town, Flat Type, Flat Model, storey_range. You may add additional rules."*

**Goal:** Run a fixed set of hand-rolled validation rules over the master, splitting it into passed/failed rows and producing a per-rule summary report. Failed rows are written to `data/failed/validation_failures.csv` with a `failure_reasons` column.

**Approach:** All allowed-value sets are *derived from the master at runtime*, never hardcoded — in line with the brief's "avoid hardcoding" directive. The split between `derive_spec()` (build the canonical sets / bounds) and `validate_master()` (apply the rules) is deliberate: it lets a *future batch* be validated against a frozen master spec without re-deriving the sets. On the master itself the categorical-set rules are tautological by construction; on a future batch they are the substantive guard.

**Rules implemented (`src/validate.py`):**

| # | Rule | Logic |
|---|---|---|
| 1 | `month_format` | matches `YYYY-MM` *and* parses with `pd.to_datetime` |
| 2 | `month_in_observed_range` | within `[master.month.min(), master.month.max()]` |
| 3 | `town_in_observed_set` | in the canonical (strip+upper) set of master towns |
| 4 | `flat_type_in_observed_set` | same pattern, derived from master |
| 5 | `flat_model_in_observed_set` | same pattern; absorbs the known "Improved" vs "IMPROVED" case drift via the canonical normalization |
| 6 | `storey_range_format` | matches `NN TO NN` with `hi > lo` |
| 7 | `storey_range_in_observed_set` | in the canonical set of master storey ranges |
| 8 | `floor_area_positive` | `> 0` |
| 9 | `floor_area_within_iqr_bounds` | within `[Q1 − 3·IQR, Q3 + 3·IQR]` from `derive_spec()`; flags structurally valid but extreme rows |
| 10 | `resale_price_positive` | `> 0` |
| 11 | `lease_commence_year_in_range` | between `LEASE_MIN_YEAR` (1960) and the current year |
| 12 | `composite_key_no_nulls` | no nulls in any column except `resale_price`, `source_file`, and the (vintage-absent) `remaining_lease*` columns |

Convention: a rule's mask is `True` for *failing* rows. Unevaluable values (NaN where the rule needs a number) are treated as failures rather than silently passing.

In [11]:
from src.validate import derive_spec, validate_master

spec = derive_spec(master)
result = validate_master(master, spec=spec)

print(f"Spec derived from master ({len(master):,} rows):")
print(f"  month range:        {spec.month_min} .. {spec.month_max}")
print(f"  towns:              {len(spec.towns)} canonical values")
print(f"  flat_types:         {len(spec.flat_types)} canonical values")
print(f"  flat_models:        {len(spec.flat_models)} canonical values")
print(f"  storey_ranges:      {len(spec.storey_ranges)} canonical values")
print(f"  floor_area band:    [{spec.floor_area_lower:.1f}, {spec.floor_area_upper:.1f}] sqm")
print()
print(f"Passed: {len(result.passed_df):,}    Failed: {len(result.failed_df):,}")

Spec derived from master (92,544 rows):
  month range:        2012-01 .. 2016-12
  towns:              26 canonical values
  flat_types:         7 canonical values
  flat_models:        20 canonical values
  storey_ranges:      25 canonical values
  floor_area band:    [-37.0, 222.0] sqm

Passed: 92,538    Failed: 6


### 4.1 Per-rule report

One row per rule: how many rows were checked, how many failed, the failure percentage, and a small sample of failing rows for diagnosis. The `sample_failures` column is hidden in the display below for readability.

In [12]:
display(
    result.report[["rule", "rows_checked", "rows_failed", "fail_pct"]]
    .style.format({"fail_pct": "{:.4%}"})
)

,rule,rows_checked,rows_failed,fail_pct
0,month_format,92544,0,0.0000%
1,month_in_observed_range,92544,0,0.0000%
2,town_in_observed_set,92544,0,0.0000%
3,flat_type_in_observed_set,92544,0,0.0000%
4,flat_model_in_observed_set,92544,0,0.0000%
5,storey_range_format,92544,0,0.0000%
6,storey_range_in_observed_set,92544,0,0.0000%
7,floor_area_positive,92544,0,0.0000%
8,floor_area_within_iqr_bounds,92544,6,0.0065%
9,resale_price_positive,92544,0,0.0000%


### 4.2 Failed rows

Rows failing one or more rules are written to `data/failed/validation_failures.csv` with a `failure_reasons` column listing every rule each row violated. The directory is created on demand.

**Expected finding on this master:** the only rule that fires is `floor_area_within_iqr_bounds` — a small handful of structurally valid but statistically extreme floor areas in the upper tail of the distribution. This is intentionally a *flag*, not a hard fail; the cleaning stage decides whether to drop them.

In [13]:
config.FAILED_DIR.mkdir(parents=True, exist_ok=True)
failed_path = config.FAILED_DIR / "validation_failures.csv"
result.failed_df.to_csv(failed_path, index=False)

print(f"Wrote {len(result.failed_df):,} failed rows to {failed_path.relative_to(config.PROJECT_ROOT)}")
print()
print("Failure reason breakdown:")
print(result.failed_df["failure_reasons"].value_counts().to_string())
print()
print("Sample failed rows:")
display(
    result.failed_df[
        ["month", "town", "flat_type", "flat_model", "storey_range",
         "floor_area_sqm", "resale_price", "failure_reasons"]
    ].head(10)
)

Wrote 6 failed rows to data/failed/validation_failures.csv

Failure reason breakdown:
failure_reasons
floor_area_within_iqr_bounds    6

Sample failed rows:


,month,town,flat_type,flat_model,storey_range,floor_area_sqm,resale_price,failure_reasons
6301,2015-03,KALLANG/WHAMPOA,3 ROOM,Terrace,01 TO 03,280.0,1060000.0,floor_area_within_iqr_bounds
11093,2015-06,KALLANG/WHAMPOA,3 ROOM,Terrace,01 TO 03,241.0,958000.0,floor_area_within_iqr_bounds
39678,2016-12,KALLANG/WHAMPOA,3 ROOM,Terrace,01 TO 03,259.0,1150000.0,floor_area_within_iqr_bounds
42962,2012-04,BISHAN,EXECUTIVE,Maisonette,06 TO 10,243.0,869000.0,floor_area_within_iqr_bounds
52454,2012-08,KALLANG/WHAMPOA,3 ROOM,Terrace,01 TO 03,249.0,988888.0,floor_area_within_iqr_bounds
65077,2013-04,KALLANG/WHAMPOA,3 ROOM,Terrace,01 TO 03,266.0,1020000.0,floor_area_within_iqr_bounds


## 5. Clean

**Brief reference:** *Data Quality Requirements §4–6.*

**Goal:** Take the validation-passing rows and produce the final cleaned dataset by:

1. **Recomputing remaining lease as of today** (DQ §4). Each HDB flat has a 99-year lease starting from `lease_commence_date`. We compute remaining months/years against `config.AS_OF_DATE` (default: today) and store the result as both an integer month count (`remaining_lease_computed_months`) and a "Y years M months" string (`remaining_lease_computed`). Rows whose recomputed lease is negative are split off as expired and excluded from the cleaned output.

2. **Deduplicating on the composite key** (DQ §5). The composite key is every column that identifies a transaction — i.e. all raw columns *except* `resale_price` (the value being deduped on), `source_file` (lineage), the vintage-specific `remaining_lease*` columns (NaN in pre-2015 vintages by design), and any clean-stage additions. On a tie we keep the row with the highest `resale_price`; the lower-price losers are written separately for review.

3. **Flagging anomalous resale prices** (DQ §6). Heuristic: compute price-per-sqm, then within each `(town, flat_type, year)` group derive Q1 and Q3. Flag rows whose price-per-sqm falls outside `[Q1 − 1.5·IQR, Q3 + 3.0·IQR]`. The bounds are **asymmetric** — tighter on the cheap side because impossibly cheap flats are more suspicious than luxury outliers (multipliers in `config.IQR_LOWER_MULT` / `IQR_UPPER_MULT`). Anomalies are *flagged*, never dropped: `price_anomaly_flag` (bool) and `price_anomaly_reason` (string) are added to the cleaned frame.

`src/clean.py` exposes a single pure entrypoint, `clean_master(passed_df)`, returning a `CleanResult` with the cleaned frame, the expired/duplicate side outputs, the anomaly subset, and per-stage row counts. The notebook handles all I/O.

In [14]:
from src.clean import clean_master, composite_key_columns

clean_result = clean_master(result.passed_df)

print(f"Composite key ({len(composite_key_columns(result.passed_df))} cols):")
for c in composite_key_columns(result.passed_df):
    print(f"  - {c}")
print()
print(f"as_of_date: {config.AS_OF_DATE}")
print(f"Cleaned frame shape: {clean_result.cleaned_df.shape[0]:,} rows x {clean_result.cleaned_df.shape[1]} columns")

Composite key (9 cols):
  - month
  - town
  - flat_type
  - block
  - street_name
  - storey_range
  - floor_area_sqm
  - flat_model
  - lease_commence_date

as_of_date: 2026-04-09
Cleaned frame shape: 90,938 rows x 17 columns


### 5.1 Side outputs — write expired, duplicate-loser, and anomaly-flag files

Three side files are written:

* `data/failed/lease_expired.csv` — rows whose recomputed lease is negative (removed from cleaned output)
* `data/failed/duplicate_lower_price.csv` — losers from the dedupe step (removed from cleaned output, per brief DQ §5)
* `data/reports/price_anomalies_flagged.csv` — rows flagged by the price anomaly heuristic (**also retained** in the cleaned output)

The split between `failed/` and `reports/` is deliberate. The brief defines `Failed` as *"records that were **removed**"* (validation failures, duplicates). DQ §6 anomalies are explicitly *flagged*, not removed — the wording is *"identify any potentially anomalous resale price"*, with no "discard" verb anywhere in §6 — so they live under `reports/` rather than `failed/`.

In [15]:
config.FAILED_DIR.mkdir(parents=True, exist_ok=True)
config.REPORTS_DIR.mkdir(parents=True, exist_ok=True)

expired_path = config.FAILED_DIR / "lease_expired.csv"
duplicates_path = config.FAILED_DIR / "duplicate_lower_price.csv"
anomalies_path = config.REPORTS_DIR / "price_anomalies_flagged.csv"

clean_result.expired_df.to_csv(expired_path, index=False)
clean_result.duplicate_losers_df.to_csv(duplicates_path, index=False)
clean_result.anomalies_df.to_csv(anomalies_path, index=False)

print(f"Wrote {len(clean_result.expired_df):>6,d} expired rows to {expired_path.relative_to(config.PROJECT_ROOT)}")
print(f"Wrote {len(clean_result.duplicate_losers_df):>6,d} duplicate losers to {duplicates_path.relative_to(config.PROJECT_ROOT)}")
print(f"Wrote {len(clean_result.anomalies_df):>6,d} flagged anomalies to {anomalies_path.relative_to(config.PROJECT_ROOT)}")
print()
print("Sample anomaly rows:")
display(
    clean_result.anomalies_df[
        ["month", "town", "flat_type", "floor_area_sqm", "resale_price", "price_anomaly_reason"]
    ].head(8)
)

Wrote      0 expired rows to data/failed/lease_expired.csv
Wrote  1,600 duplicate losers to data/failed/duplicate_lower_price.csv
Wrote    623 flagged anomalies to data/reports/price_anomalies_flagged.csv

Sample anomaly rows:


,month,town,flat_type,floor_area_sqm,resale_price,price_anomaly_reason
4,2012-01,ANG MO KIO,2 ROOM,45.0,226000.0,below_lower_iqr_bound
321,2012-01,BUKIT PANJANG,3 ROOM,73.0,301000.0,below_lower_iqr_bound
567,2012-01,HOUGANG,3 ROOM,80.0,325000.0,below_lower_iqr_bound
571,2012-01,HOUGANG,3 ROOM,82.0,340000.0,below_lower_iqr_bound
592,2012-01,HOUGANG,4 ROOM,100.0,334000.0,below_lower_iqr_bound
614,2012-01,HOUGANG,5 ROOM,125.0,442888.0,below_lower_iqr_bound
629,2012-01,HOUGANG,EXECUTIVE,152.0,475000.0,below_lower_iqr_bound
690,2012-01,JURONG WEST,3 ROOM,67.0,246000.0,below_lower_iqr_bound


Final cleaned dataset written to `data/cleaned/hdb_resale_cleaned.csv`. 

In [16]:
config.CLEANED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_csv = config.CLEANED_DIR / "hdb_resale_cleaned.csv"

clean_result.cleaned_df.to_csv(cleaned_csv, index=False)

print(f"Wrote {len(clean_result.cleaned_df):,} rows to:")
print(f"  {cleaned_csv.relative_to(config.PROJECT_ROOT)}  ({cleaned_csv.stat().st_size:,} bytes)")
print()
print("First few rows of the cleaned dataset:")
display(clean_result.cleaned_df.head())

Wrote 90,938 rows to:
  data/cleaned/hdb_resale_cleaned.csv  (16,442,943 bytes)

First few rows of the cleaned dataset:


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,remaining_lease,remaining_lease_years_original,remaining_lease_computed_months,remaining_lease_computed,price_anomaly_flag,price_anomaly_reason
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>,620,51 years 8 months,False,<NA>
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>,608,50 years 8 months,False,<NA>
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>,608,50 years 8 months,False,<NA>
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>,704,58 years 8 months,False,<NA>
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>,704,58 years 8 months,True,below_lower_iqr_bound


### 5.3 Pipeline row-count summary

How many rows survived each stage of the pipeline, end to end. The numbers below trace a single transaction's journey from the raw CSVs (where most rows are out of scope) through scope filtering, validation, and cleaning.

In [17]:
import pandas as _pd

raw_total = sum(r["rows"] for r in verification)

stage_summary = _pd.DataFrame(
    [
        {"stage": "1. ingest (raw rows in all 3 in-scope CSVs)",     "rows": raw_total,                        "delta": None},
        {"stage": "2. combine (after scope filter to 2012-01..2016-12)", "rows": len(master),                 "delta": len(master) - raw_total},
        {"stage": "3. validate.passed",                                "rows": len(result.passed_df),          "delta": len(result.passed_df) - len(master)},
        {"stage": "4a. clean.recompute_lease (after dropping expired)", "rows": clean_result.stage_counts["after_lease_recompute"], "delta": -clean_result.stage_counts["expired_lease_dropped"]},
        {"stage": "4b. clean.deduplicate (after dropping lower-price duplicates)", "rows": clean_result.stage_counts["after_dedupe"], "delta": -clean_result.stage_counts["duplicates_dropped"]},
        {"stage": "5. cleaned output (anomalies flagged, not dropped)", "rows": clean_result.stage_counts["final_cleaned_rows"], "delta": 0},
    ]
)
display(stage_summary)

print()
print(f"Anomalies flagged in cleaned output: {clean_result.stage_counts['anomalies_flagged']:,}"
      f" ({clean_result.stage_counts['anomalies_flagged'] / max(1, clean_result.stage_counts['final_cleaned_rows']):.2%})")

,stage,rows,delta
0,1. ingest (raw rows in all 3 in-scope CSVs),459007,NaN
1,2. combine (after scope filter to 2012-01..201...,92544,-366463.0
2,3. validate.passed,92538,-6.0
3,4a. clean.recompute_lease (after dropping expi...,92538,0.0
4,4b. clean.deduplicate (after dropping lower-pr...,90938,-1600.0
5,"5. cleaned output (anomalies flagged, not drop...",90938,0.0



Anomalies flagged in cleaned output: 623 (0.69%)


## 6. Transform

**Brief reference:** *Data Transformation Requirements §1–3.*

**Goal:** Take the cleaned dataset and produce the Transformed and Hashed output groups.

1. Construct a 9-character `resale_identifier` for each row from five components (literal `"S"`, 3 block digits, 2 average-price digits, 2 month digits, 1 town initial).
2. Deduplicate on the identifier — because the identifier is intentionally lossy (block→3 digits, price→2 digits), genuinely different flats can collide on it. On a collision, keep the highest `resale_price` (brief Transformation §2) and route the losers to `data/failed/`.
3. Hash the identifier with SHA-256 and emit the Hashed output group.

**Interpretation note.** The brief's Transformation §2 says *"if there are any duplicate records, take the higher price and discard the lower price one."* Stage 5 (§5.2) has already deduplicated on the composite key of raw columns, so any *remaining* duplicates must come from §1 — the identifier construction itself. We therefore apply §2 at the identifier level: rows sharing a `resale_identifier` are the new duplicate class to resolve.

`src/transform.py` holds all the logic as pure functions; the notebook handles I/O.

### 6.1 Load cleaned dataset

Start from `data/cleaned/hdb_resale_cleaned.csv` — this pass does not re-run stages 1–5. The CSV round-trips the nullable Int64 / string / bool columns as plain object columns, which is fine for the transformation stage (we only touch `block`, `month`, `town`, `flat_type`, `resale_price`).

In [18]:
from src.transform import (
    build_resale_identifier,
    dedupe_by_identifier,
    add_hashed_identifier,
)

config.TRANSFORMED_DIR.mkdir(parents=True, exist_ok=True)
config.HASHED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = config.CLEANED_DIR / "hdb_resale_cleaned.csv"
cleaned = pd.read_csv(cleaned_path)
print(f"Loaded {len(cleaned):,} rows from {cleaned_path.relative_to(config.PROJECT_ROOT)}")

Loaded 90,938 rows from data/cleaned/hdb_resale_cleaned.csv


### 6.2 Build the Resale Identifier

`build_resale_identifier` merges the per-(month, town, flat_type) mean resale price onto each row, constructs the 9-character identifier, and asserts every generated value is exactly `IDENTIFIER_LENGTH` chars as a safety net against spec drift.

In [19]:
transformed = build_resale_identifier(cleaned)

print(f"Built identifiers for {len(transformed):,} rows")
print(f"All identifiers are {config.IDENTIFIER_LENGTH} chars:",
      (transformed['resale_identifier'].str.len() == config.IDENTIFIER_LENGTH).all())
print()
display(
    transformed[["block", "month", "town", "flat_type", "resale_price", "resale_identifier"]].head(10)
)

Built identifiers for 90,938 rows
All identifiers are 9 chars: True



,block,month,town,flat_type,resale_price,resale_identifier
0,406,2012-01,ANG MO KIO,2 ROOM,257800.0,S4062501A
1,314,2012-01,ANG MO KIO,2 ROOM,263000.0,S3142501A
2,314,2012-01,ANG MO KIO,2 ROOM,275000.0,S3142501A
3,170,2012-01,ANG MO KIO,2 ROOM,260000.0,S1702501A
4,174,2012-01,ANG MO KIO,2 ROOM,226000.0,S1742501A
5,508,2012-01,ANG MO KIO,2 ROOM,260000.0,S5082501A
6,174,2012-01,ANG MO KIO,3 ROOM,281000.0,S1743401A
7,216,2012-01,ANG MO KIO,3 ROOM,375000.0,S2163401A
8,332,2012-01,ANG MO KIO,3 ROOM,350000.0,S3323401A
9,418,2012-01,ANG MO KIO,3 ROOM,420000.0,S4183401A


### 6.3 Identifier-collision diagnostics

The identifier is intentionally lossy, so collisions are expected. A high collision rate is not a bug — it's a property of the construction spec. The interesting diagnostics:

* **Distinct identifiers** vs **total rows** — the compression ratio.
* **Top collided identifiers** — which identifiers attract the most collisions, and how wide is the price range within a single collision bucket? A wide range is the reviewer-visible proof that these are genuinely different flats, not latent duplicates the cleaning stage missed.

In [20]:
n_total = len(transformed)
n_distinct = transformed["resale_identifier"].nunique()
id_counts = transformed["resale_identifier"].value_counts()
n_collided_ids = int((id_counts > 1).sum())
n_rows_in_collisions = int(id_counts[id_counts > 1].sum())

print(f"Total rows:                  {n_total:>8,d}")
print(f"Distinct identifiers:        {n_distinct:>8,d}")
print(f"Rows per identifier (mean):  {n_total / n_distinct:>8.2f}")
print(f"Identifiers with collisions: {n_collided_ids:>8,d}  ({n_collided_ids / n_distinct:.1%} of distinct ids)")
print(f"Rows in collided buckets:    {n_rows_in_collisions:>8,d}  ({n_rows_in_collisions / n_total:.1%} of all rows)")
print()
print("Top 5 most-collided identifiers (with intra-bucket price range):")

top_ids = id_counts.head(5).index
collision_report = (
    transformed[transformed["resale_identifier"].isin(top_ids)]
    .groupby("resale_identifier")["resale_price"]
    .agg(rows="count", min_price="min", max_price="max")
    .sort_values("rows", ascending=False)
)
collision_report["price_spread"] = collision_report["max_price"] - collision_report["min_price"]
display(collision_report)

Total rows:                    90,938
Distinct identifiers:          77,246
Rows per identifier (mean):      1.18
Identifiers with collisions:   10,951  (14.2% of distinct ids)
Rows in collided buckets:      24,643  (27.1% of all rows)

Top 5 most-collided identifiers (with intra-bucket price range):


,rows,min_price,max_price,price_spread
resale_identifier,,,,
S0018406C,11,650000.0,965000.0,315000.0
S4365104S,11,510000.0,563000.0,53000.0
S0018302C,9,810000.0,950000.0,140000.0
S0018305C,9,795000.0,900000.0,105000.0
S3014303P,9,416888.0,526888.0,110000.0


### 6.4 Dedupe on identifier, route losers to `failed/`

Brief Transformation §2: keep the highest-price row per identifier, discard the rest. Losers are written to `data/failed/identifier_collision_duplicates.csv` with a `failure_reason` column so the file slots into the existing `failed/` convention.

In [21]:
transformed_kept, collision_losers = dedupe_by_identifier(transformed)

collision_losers_path = config.FAILED_DIR / "identifier_collision_duplicates.csv"
collision_losers.to_csv(collision_losers_path, index=False)

print(f"Kept:      {len(transformed_kept):>8,d} rows (1 per distinct identifier)")
print(f"Discarded: {len(collision_losers):>8,d} rows (lower-price collision losers)")
print(f"Wrote discarded rows to {collision_losers_path.relative_to(config.PROJECT_ROOT)}")

Kept:        77,246 rows (1 per distinct identifier)
Discarded:   13,692 rows (lower-price collision losers)
Wrote discarded rows to data/failed/identifier_collision_duplicates.csv


### 6.5 Write Transformed output

`data/transformed/hdb_resale_transformed.csv` = cleaned schema + `resale_identifier`, post-collision-dedupe, **without** the hash. The hash lives in the separate Hashed output group below.

In [22]:
transformed_path = config.TRANSFORMED_DIR / "hdb_resale_transformed.csv"
transformed_kept.to_csv(transformed_path, index=False)

print(f"Wrote {len(transformed_kept):,} rows to {transformed_path.relative_to(config.PROJECT_ROOT)}"
      f" ({transformed_path.stat().st_size:,} bytes)")
print()
display(transformed_kept[["month", "town", "flat_type", "block", "resale_price", "resale_identifier"]].head())

Wrote 77,246 rows to data/transformed/hdb_resale_transformed.csv (14,793,500 bytes)



,month,town,flat_type,block,resale_price,resale_identifier
0,2012-01,ANG MO KIO,2 ROOM,406,257800.0,S4062501A
2,2012-01,ANG MO KIO,2 ROOM,314,275000.0,S3142501A
3,2012-01,ANG MO KIO,2 ROOM,170,260000.0,S1702501A
4,2012-01,ANG MO KIO,2 ROOM,174,226000.0,S1742501A
5,2012-01,ANG MO KIO,2 ROOM,508,260000.0,S5082501A


### 6.6 Add the hashed identifier (SHA-256)

`add_hashed_identifier` maps every `resale_identifier` through SHA-256 and asserts `nunique(hashed) == nunique(plaintext)` at runtime to verify uniqueness preservation. At 256-bit output and ~77k distinct identifiers the collision probability is ~10⁻⁶⁴ by the birthday bound, so a mismatch would almost certainly indicate a pipeline bug rather than a real collision.

In [23]:
hashed_full = add_hashed_identifier(transformed_kept)

print(f"Hashed {len(hashed_full):,} identifiers")
print(f"Distinct plaintext:  {hashed_full['resale_identifier'].nunique():,}")
print(f"Distinct hashes:     {hashed_full['resale_identifier_hashed'].nunique():,}")
print()
display(
    hashed_full[["resale_identifier", "resale_identifier_hashed"]].head()
)

Hashed 77,246 identifiers
Distinct plaintext:  77,246
Distinct hashes:     77,246



,resale_identifier,resale_identifier_hashed
0,S4062501A,e22d10c12612bb3c3d1ff735a675db47039477c6ce8dbe...
2,S3142501A,df36be696d8b9d99bff955089fa0d1ee5cc92ae167f78a...
3,S1702501A,7b76c2e5ad415d73d9ac4e74b364f51dc0b60f0db59b52...
4,S1742501A,0509ae6467b1e75c84fe77c74c7f8857f8926009d7b912...
5,S5082501A,2fc7d3dd15e6bbb2178269cc278c2a995c10378f770d5d...


### 6.7 Write Hashed output

Per the brief the Hashed output group is *"Cleaned Data + Hashed Identifier Column"*. We therefore drop the plaintext `resale_identifier` before writing — the whole point of hashing is that the plaintext is not exposed alongside the hash.

In [24]:
hashed_out = hashed_full.drop(columns=["resale_identifier"])

hashed_path = config.HASHED_DIR / "hdb_resale_hashed.csv"
hashed_out.to_csv(hashed_path, index=False)

print(f"Wrote {len(hashed_out):,} rows to {hashed_path.relative_to(config.PROJECT_ROOT)}"
      f" ({hashed_path.stat().st_size:,} bytes)")
print()
print("Hashed output columns:")
for c in hashed_out.columns:
    marker = "  <-- hash" if c == "resale_identifier_hashed" else ""
    print(f"  {c}{marker}")

Wrote 77,246 rows to data/hashed/hdb_resale_hashed.csv (19,042,037 bytes)

Hashed output columns:
  month
  town
  flat_type
  block
  street_name
  storey_range
  floor_area_sqm
  flat_model
  lease_commence_date
  resale_price
  source_file
  remaining_lease
  remaining_lease_years_original
  remaining_lease_computed_months
  remaining_lease_computed
  price_anomaly_flag
  price_anomaly_reason
  resale_identifier_hashed  <-- hash


### 6.8 End-to-end pipeline row counts

Full lineage from raw CSVs through to the Hashed output group.

In [25]:
raw_total = sum(r["rows"] for r in verification)

pipeline_summary = pd.DataFrame(
    [
        {"stage": "1. ingest (raw rows, all 3 in-scope CSVs)",             "rows": raw_total},
        {"stage": "2. combine (after scope filter to 2012-01..2016-12)",    "rows": len(master)},
        {"stage": "3. validate.passed",                                     "rows": len(result.passed_df)},
        {"stage": "4. clean.cleaned (composite-key dedupe, anomalies flagged)", "rows": len(clean_result.cleaned_df)},
        {"stage": "5. transform.with_identifier (pre collision dedupe)",     "rows": len(transformed)},
        {"stage": "6. transform.kept (post identifier-collision dedupe)",    "rows": len(transformed_kept)},
        {"stage": "7. hashed (same row count, plaintext id dropped)",        "rows": len(hashed_out)},
    ]
)
display(pipeline_summary)

,stage,rows
0,"1. ingest (raw rows, all 3 in-scope CSVs)",459007
1,2. combine (after scope filter to 2012-01..201...,92544
2,3. validate.passed,92538
3,"4. clean.cleaned (composite-key dedupe, anomal...",90938
4,5. transform.with_identifier (pre collision de...,90938
5,6. transform.kept (post identifier-collision d...,77246
6,"7. hashed (same row count, plaintext id dropped)",77246


## Done

All pipeline stages of Part 1 are now implemented end-to-end:

| Stage | Module | Output |
|---|---|---|
| 1. Ingest | `src/ingest.py` | `data/raw/*.csv` |
| 2. Combine | `src/combine.py` | in-memory `master` DataFrame |
| 3. Profile | `src/profile.py` | in-notebook `ProfileReport` |
| 4. Validate | `src/validate.py` | `data/failed/validation_failures.csv` |
| 5. Clean | `src/clean.py` | `data/cleaned/hdb_resale_cleaned.csv` plus side files in `data/failed/` and `data/reports/` |
| 6. Transform | `src/transform.py` | `data/transformed/hdb_resale_transformed.csv`, `data/hashed/hdb_resale_hashed.csv`, `data/failed/identifier_collision_duplicates.csv` |

**End of Technical Test Part 1**